# Chapter 19: Custom Actions: Composite, JavaScript, Docker (notebook edition)

## Learning Objectives

- Run and test a composite action
- Explain JavaScript action behavior
- Validate action metadata beyond actionlint

In [ ]:
import os, sys
from pathlib import Path

ROOT = Path.cwd().resolve()
while not (ROOT / "pyproject.toml").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

GHA_MODE = os.environ.get("GHA_MODE", "fixture")  # "fixture" | "live"
GHA_REPO = os.environ.get("GHA_REPO", "Friend09/practice_git_intro_to_github_actions")
print("mode:", GHA_MODE, "| repo:", GHA_REPO)

## 1. Run the composite action's script

The script reads `VERSION` and writes outputs to the file named by `GITHUB_OUTPUT`, exactly as on the runner. We compare it with the Python SemVer parser.

In [ ]:
import sys
sys.path.insert(0, str(ROOT / "labs"))
import lab_19_custom_actions as L
from intro_gha.semver import parse
for v in ["0.1.0", "1.2.3-rc.1", "1.2", "v1.2.3", "01.2.3"]:
    code, out = L.run_script(v)
    print(f"{v:12} exit={code} {out}")
assert L.run_script("1.2.3-rc.1")[1]["is_prerelease"] == "true" and L.run_script("1.2")[0] != 0

## 2. What the real run printed

Valid input produced outputs; invalid input failed with empty outputs; a remote path reference worked without a checkout.

In [ ]:
import json
run = json.loads((ROOT/"fixtures"/"actions_ch19_run.json").read_text())["reports"]
print(run["composite"]); print(run["remote_reference"])
assert run["composite"]["bad_outcome"] == "failure" and run["remote_reference"]["remote_ref_parts"] == "3.4.5"

## 3. The JavaScript action

Hyphenated inputs keep their hyphens, the Node version is the runner's, and `post` ran even after a later step failed.

In [ ]:
js = run["javascript"]; print(js)
print(run["javascript_post_after_failure"])
assert js["js_inputs"] == "INPUT_SHOUT,INPUT_WHO-TO-GREET" and run["javascript_post_after_failure"]["js_post_ran"] == "yes"

## 4. Docker: build vs prebuilt

A Dockerfile action rebuilds every run.

In [ ]:
s = json.loads((ROOT/"fixtures"/"actions_ch19_summary.json").read_text())
print(s["docker"]["built"]["step_seconds"], "s built vs", s["docker"]["prebuilt"]["step_seconds"], "s prebuilt;", 40 * 200, "s of build per day at 40 s x 200 uses")
assert s["docker"]["built"]["step_seconds"] > s["docker"]["prebuilt"]["step_seconds"]

## 5. Validate action metadata yourself

`actionlint` passed an action with a composite `run:` step missing `shell`; GitHub rejected it. Our check catches it.

In [ ]:
bad = L.metadata("tally-noshell")
missing = [st for st in bad["runs"]["steps"] if "run" in st and "shell" not in st]
print("steps missing shell:", len(missing), "| actionlint exit code was", s["broken_actions"]["no_shell"]["actionlint_exit"])
print(s["broken_actions"]["no_shell"]["github_error"][-60:])
assert len(missing) == 1

## Takeaways & Next Steps

- Composite for reuse, JavaScript for logic, Docker for environments.
- Unit-test, metadata-test and workflow-test.
- Next: Chapter 20, reusable workflows.

## Chapter Link

Read: `learning_modules/chapter_19_custom_actions.md`